# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 05: Link Prediction & Knowledge Graph Embeddings (TransE)

### 1. Conceptual Intuition & Real-World Use Case
Beyond classifying individual nodes, many real-world problems require predicting whether a relationship (edge) exists between two entities:
- **Biomedical Drug Repurposing:** Will Molecule $X$ inhibit Disease Target $Y$?
- **Knowledge Graphs:** Given (Barack Obama, BornIn, ?), predict Hawaii.
- **Recommender Systems:** Will User $A$ purchase Item $B$?

**TransE (Translational Embeddings)** by Bordes et al. models relationships as geometric translations in vector space:
$$\mathbf{head} + \mathbf{relation} \approx \mathbf{tail}$$

---

### 2. Mathematical Foundations
A knowledge graph fact is a triplet $(h, r, t) \in E$:
- $h \in \mathbb{R}^k$: Head entity vector
- $r \in \mathbb{R}^k$: Relation vector
- $t \in \mathbb{R}^k$: Tail entity vector

1. **Distance Energy Function:**
   $$d(h, r, t) = \|h + r - t\|_{L_1 \text{ or } L_2}$$
   If the relationship is true, $d(h, r, t) \approx 0$.

2. **Margin-Based Ranking Loss:**
   $$\mathcal{L} = \sum_{(h, r, t) \in S} \sum_{(h', r, t') \in S'} \max\left(0, \gamma + d(h, r, t) - d(h', r, t')\right)$$
   Where:
   - $\gamma > 0$ is the margin separation hyperparameter.
   - $S$ is the set of true valid facts.
   - $S'$ is the set of corrupted negative facts (where either head or tail is replaced with a random entity).
   - Constraint: $\|h\|_2 = 1, \|t\|_2 = 1$ to prevent trivial zero-collapse.

3. **Evaluation Metrics:**
   - **Hits@10:** Percentage of test triplets where the true entity ranks in the top 10.
   - **Mean Reciprocal Rank (MRR):** $\frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{Rank}_i}$.

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Define Entities and Relations in a Molecular / Chemical Graph
entities = ["Aspirin", "Paracetamol", "COX1_Protein", "Pain_Receptor", "Liver_Enzyme"]
relations = ["inhibits", "metabolized_by", "activates"]

entity_to_id = {e: i for i, e in enumerate(entities)}
relation_to_id = {r: i for i, r in enumerate(relations)}

num_entities = len(entities)
num_relations = len(relations)
dim = 4  # Embedding dimension

# Positive true triplets (h, r, t)
positive_triplets = [
    (entity_to_id["Aspirin"], relation_to_id["inhibits"], entity_to_id["COX1_Protein"]),
    (entity_to_id["Paracetamol"], relation_to_id["metabolized_by"], entity_to_id["Liver_Enzyme"]),
    (entity_to_id["Aspirin"], relation_to_id["inhibits"], entity_to_id["Pain_Receptor"])
]

# Step 2: Initialize entity and relation embeddings
E = np.random.uniform(-0.5, 0.5, (num_entities, dim))
R = np.random.uniform(-0.5, 0.5, (num_relations, dim))

# Normalize entity embeddings to unit sphere
E = E / np.linalg.norm(E, axis=1, keepdims=True)

def distance(h, r, t):
    # L2 distance ||h + r - t||
    return np.linalg.norm(h + r - t)

# Step 3: TransE Training with Margin Ranking Loss
gamma = 1.0   # Margin
lr = 0.05
epochs = 120

for epoch in range(epochs):
    epoch_loss = 0.0
    for h_id, r_id, t_id in positive_triplets:
        h = E[h_id]
        r = R[r_id]
        t = E[t_id]
        d_pos = distance(h, r, t)
        
        # Sample negative corrupted triplet (corrupt tail)
        t_neg_id = np.random.choice([x for x in range(num_entities) if x != t_id])
        t_neg = E[t_neg_id]
        d_neg = distance(h, r, t_neg)
        
        # Loss: max(0, gamma + d_pos - d_neg)
        loss = max(0.0, gamma + d_pos - d_neg)
        epoch_loss += loss
        
        if loss > 0:
            # Gradient updates
            grad_pos = (h + r - t) / (d_pos + 1e-8)
            grad_neg = (h + r - t_neg) / (d_neg + 1e-8)
            
            # Update embeddings
            E[h_id] -= lr * (grad_pos - grad_neg)
            R[r_id] -= lr * (grad_pos - grad_neg)
            E[t_id] -= lr * (-grad_pos)
            E[t_neg_id] -= lr * (grad_neg)
            
            # Re-normalize entities
            E[h_id] /= np.linalg.norm(E[h_id])
            E[t_id] /= np.linalg.norm(E[t_id])
            E[t_neg_id] /= np.linalg.norm(E[t_neg_id])

print(f"Training completed. Final Epoch Loss: {epoch_loss:.4f}")

# Step 4: Link Prediction Evaluation
# Query: (Aspirin, inhibits, ?)
query_h = entity_to_id["Aspirin"]
query_r = relation_to_id["inhibits"]

print(f"\nQuery: ({entities[query_h]}, {relations[query_r]}, ?)")
scores = []
for candidate_t in range(num_entities):
    d = distance(E[query_h], R[query_r], E[candidate_t])
    scores.append((candidate_t, d))

# Sort candidates by lowest distance
scores.sort(key=lambda x: x[1])

print("Candidate Ranking (Lowest translation distance = most likely):")
for rank, (cand_id, dist) in enumerate(scores, 1):
    print(f"  Rank {rank}: {entities[cand_id]:<15} (Distance: {dist:.4f})")

### 3. Step-by-Step Code Walkthrough
1. **Translational Principle:** $h + r \approx t$. If Aspirin inhibits COX1, then $\mathbf{Aspirin} + \mathbf{inhibits}$ should be close to $\mathbf{COX1}$.
2. **Negative Sampling:** Corrupting the tail entity with random entities forces the optimizer to increase distance $d_{neg}$ while decreasing $d_{pos}$.
3. **Link Prediction Ranking:** Given a query $(h, r, ?)$, we compute translation distances across all entities in the database and rank candidates.

---

### 4. Industry Applications & Trade-offs
- **Biomedical Discovery:** Identifying drug-drug adverse interactions before clinical trials.
- **Search Engines:** Google Knowledge Graph powering structured entity carousels.
- **Trade-off:** TransE struggles with 1-to-many, many-to-1, and symmetric relations (e.g., if $A$ is married to $B$ and $B$ is married to $A$, TransE implies $r=0$). RotatE and DistMult resolve this by using complex rotations.

---

### 5. Top Interview Questions & Answers
**Q1: Why does TransE fail on 1-to-N relationships (e.g., MotherOf)?**
> **Answer:** If $h$ has relation $r$ with both $t_1$ and $t_2$, TransE tries to satisfy $h + r = t_1$ and $h + r = t_2$. This forces $t_1 \approx t_2$, collapsing distinct entities into the same embedding point.

**Q2: What is the Mean Reciprocal Rank (MRR) metric?**
> **Answer:** For a set of test queries, $\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{Rank}_i}$. If the true entity is ranked 1st, its score is 1.0; if ranked 2nd, 0.5; if ranked 10th, 0.1. Unlike Hits@K, MRR rewards improvements across the entire rank spectrum.